# 03 — Model Training and Evaluation

This notebook executes the training pipeline, displays validation metrics, and loops through the generated analysis plots.

In [ ]:
import subprocess
result = subprocess.run(['python', '../src/train.py'], capture_output=True, text=True, cwd="..")
print("STDOUT:")
print(result.stdout)
print("STDERR:")
print(result.stderr)

In [ ]:
import json
import pandas as pd

with open('../data/results/training_results.json', 'r') as f:
    results = json.load(f)
    
results_df = pd.DataFrame(results)
display(results_df)

In [ ]:
model_comp = pd.read_csv('../data/results/model_comparison.csv', index_col=0)
display(model_comp.style.background_gradient(cmap='Blues'))

In [ ]:
# Wait, evaluate.py needs to be run to generate plots
print("Running evaluate.py...")
result_eval = subprocess.run(['python', '../src/evaluate.py'], capture_output=True, text=True, cwd="..")
print(result_eval.stdout)
print(result_eval.stderr)

In [ ]:
from IPython.display import Image, display as ipy_display
import glob

png_files = sorted(glob.glob('../data/results/*.png'))
for img_path in png_files:
    print(f"\n--- Displaying: {img_path} ---")
    ipy_display(Image(filename=img_path))

## LOMO-CV (Leave-One-Material-Out Cross Validation)

In materials science machine learning, standard random K-fold cross-validation often suffers from severe data leakage. Since certain materials (e.g., TiO2, which comprises 72% of this dataset) are heavily represented, standard splits random-assign identical host structures to both train and validation subsets. This inflates model performance metrics while failing to capture the model's true out-of-distribution generalization capabilities.

**LOMO-CV** resolves this by grouping observations by their `host_material` and reserving an entire host family (e.g. holding out all CdS catalysts or ZnO catalysts) during each validation fold. This simulates the discovery of entirely new catalyst formulations, providing a robust, publication-grade benchmark of generalizability.

## SHAP (SHapley Additive exPlanations) for Materials Insight

While XGBoost and LightGBM provide high prediction accuracy, they act as complex black boxes. **TreeSHAP** provides local, game-theoretic attributions for each feature value per prediction.

The SHAP summary and dependence plots reveal physical rules learned by the model:
- How the weight percentage of the co-catalyst affects predictions.
- The threshold boundaries of pH and loading weights.
- The relative impact of bandgap properties and light power.

## Per-Material Error Breakdown

Evaluating a model with global metrics (like overall MAE or R2) masks localized predictive failures. A material-level error analysis (reported in `per_material_metrics.csv` and plotted in `per_material_error.png`) allows researchers to identify which host classes suffer from poor accuracy. High MAE or negative R2 on a specific material type indicates that the training set lacks representative features or samples for that class, serving as a flag to direct experimental efforts towards collecting more data for those material domains.